In [1]:
import csv
from pathlib import Path

def save_synth_data(part, dim, kernel, stride, padding, ch_in, ch_out, quantization, resource_report, timing_report):

    file = Path("comparativo_CONV_2D") / "conv_2d_synth_results.csv"

    save_data = [part, dim, kernel, stride, padding, ch_in, ch_out, f"<{quantization[0]}, {quantization[1]}>",
                 resource_report["BRAM_18K"], resource_report["DSP"], resource_report["FF"], resource_report["LUT"], resource_report["URAM"],
                 timing_report["avg_total_cycles"]]

    if not file.exists():
        with open(file, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["part", "dim", "kernel", "stride", "padding", "ch_in", "ch_out", "quantization", "BRAM_18K", "DSP", "FF", "LUT", "URAM", "Avg. cycles"])

    with open(file, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(save_data)

In [2]:
from pathlib import Path

def truncar_apos_marcador(caminho, marcador="//AQUI", manter_marcador=True):
    caminho = Path(caminho)

    if not caminho.is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")

    with open(caminho, "r", encoding="utf-8") as f:
        linhas = f.readlines()

    novas_linhas = []

    for linha in linhas:
        if marcador in linha:
            if manter_marcador:
                novas_linhas.append(linha)
            break
        novas_linhas.append(linha)

    with open(caminho, "w", encoding="utf-8") as f:
        f.writelines(novas_linhas)

In [3]:
from pathlib import Path

def append_ao_arquivo(caminho, texto, nova_linha=True):
    caminho = Path(caminho)

    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")

    if not caminho.is_file():
        raise ValueError(f"O caminho não é um arquivo: {caminho}")

    # Define se adiciona quebra de linha automaticamente
    conteudo = texto + ("\n" if nova_linha else "")

    with open(caminho, "a", encoding="utf-8") as f:
        f.write(conteudo)

In [4]:
def extract_neuron_param_code(mat, num_tabs = 0):

    if len(mat.shape) == 1:

        s = "\t" * num_tabs + "{"

        for i in range(mat.shape[0]):
            
            s += f"{mat[i]:.10f}"

            if i < mat.shape[0] - 1:
                s += ", "

        return s + "}"
    
    s = "\t" * num_tabs + "{"

    for i in range(mat.shape[0]):

        s += "\n" + extract_neuron_param_code(mat[i], num_tabs + 1)

        if i < mat.shape[0] - 1:
            s += ",\n"

    s += "\n" + "\t" * num_tabs + "}"
    return s

In [5]:
import numpy as np

def gerar_tensor(shape, seed=42):
    np.random.seed(seed)
    return np.random.rand(*shape) - 0.5

# tensor = gerar_tensor((3, 2))
# print(f"quant_t weights_1[{1}][{2}][{3}][{4}] = {extract_neuron_param_code(tensor)}")

In [6]:
def generate_code(dim, kernel, stride, padding, ch_in, ch_out, quantization):

    truncar_apos_marcador("comparativo_CONV_2D/vitis/snn_implementation.cpp")
    truncar_apos_marcador("comparativo_CONV_2D/vitis/snn_implementation.h")
    truncar_apos_marcador("comparativo_CONV_2D/vitis/neuron_params.h")
    truncar_apos_marcador("comparativo_CONV_2D/vitis/quantization.h")

    # snn_implementation.h

    import math
    dim_out = math.floor((dim + 2 * padding - kernel) / stride) + 1

    hearder = f"void snn_to_hls(quant_t (&input)[{ch_in}][{dim}][{dim}], quant_t (&output)[{ch_out}][{dim_out}][{dim_out}])"
    append_ao_arquivo("comparativo_CONV_2D/vitis/snn_implementation.h", hearder + ";")

    # snn_implementation.cpp

    code = hearder + "\n{\n"
    code += f"\tConv2d<{kernel}, {kernel}, {stride}, {stride}, {padding}, {padding}, 1, 1, 1>(input, output, weights_1, bias_1);\n{'}'}"

    append_ao_arquivo("comparativo_CONV_2D/vitis/snn_implementation.cpp", code)

    # quantization.h

    ap_fixed = f"ap_fixed<{quantization[0]}, {quantization[1]}>"

    quant_code = f"typedef {ap_fixed} quant_t;"

    append_ao_arquivo("comparativo_CONV_2D/vitis/quantization.h", quant_code)

    # neuron_params.h

    weight = gerar_tensor((ch_out, ch_in, kernel, kernel))
    bias = gerar_tensor((ch_out,))

    params_code = f"quant_t weights_1[{ch_out}][{ch_in}][{kernel}][{kernel}] = {extract_neuron_param_code(weight)};\n"
    params_code += f"quant_t bias_1[{ch_out}] = {extract_neuron_param_code(bias)};\n"

    append_ao_arquivo("comparativo_CONV_2D/vitis/neuron_params.h", params_code)

generate_code(34, 5, 2, 1, 2, 16, (16, 8))

In [7]:
from neuro_hls import *

def get_syntesis_results(dim, kernel, stride, padding, ch_in, ch_out, quantization):
    
    FPGA_PART = "xcu250-figd2104-2L-e"

    generate_code(dim, kernel, stride, padding, ch_in, ch_out, quantization)

    neuro_hls = NeuroHls("comparativo_CONV_2D/vitis")
    neuro_hls.run_synth(frequency_MHz=100, part=FPGA_PART)
    resources = neuro_hls.get_synth_resource_usage()
    performance = neuro_hls.get_synth_performance_estimates()

    save_synth_data(FPGA_PART, dim, kernel, stride, padding, ch_in, ch_out, quantization, resources, performance)

In [10]:
get_syntesis_results(dim = 34,
                     kernel = 5,
                     stride = 2,
                     padding = 1,
                     ch_in = 2,
                     ch_out = 16,
                     quantization = (16, 8))


****** Vitis HLS - High-Level Synthesis from C, C++ and OpenCL v2024.2 (64-bit)
  **** SW Build 5238294 on Nov  8 2024
  **** IP Build 5239520 on Sun Nov 10 16:12:51 MST 2024
  **** SharedData Build 5239561 on Fri Nov 08 14:39:27 MST 2024
  **** Start of session at: Wed Mar 25 19:02:27 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2024 Advanced Micro Devices, Inc. All Rights Reserved.

source /tools/Xilinx/Vitis/2024.2/scripts/vitis_hls/hls.tcl -notrace
INFO: [HLS 200-10] For user 'user3' on host 'user3-Z690-PG-Riptide' (Linux_x86_64 version 6.11.0-28-generic) on Wed Mar 25 19:02:27 -03 2026
INFO: [HLS 200-10] On os Ubuntu 24.04.3 LTS
INFO: [HLS 200-10] In directory '/home/user3/Documentos/Fernando/NeuroHLS_dev_nir_to_cpp/NeuroHLS_dev/comparativo_CONV_2D/vitis'
Sourcing Tcl script '2_synth.tcl'
INFO: [HLS 200-1510] Running: source 2_synth.tcl
INFO: [HLS 200-1510] Running: open_project vitis_proj 
INFO: [HLS 200-10] Opening project '/home/user3

In [11]:
from tabulate import tabulate
import pandas as pd

def make_table_from_report(file_name):

    df = pd.read_csv(file_name)
    print(tabulate(df, headers="keys", tablefmt="grid", showindex=False))

make_table_from_report("comparativo_CONV_2D/conv_2d_synth_results.csv")

+----------------------+-------+----------+----------+-----------+---------+----------+----------------+------------+-------+------+-------+--------+---------------+
| part                 |   dim |   kernel |   stride |   padding |   ch_in |   ch_out | quantization   |   BRAM_18K |   DSP |   FF |   LUT |   URAM |   Avg. cycles |
+======================+=======+==========+==========+===========+=========+==========+================+============+=======+======+=======+========+===============+
| xcu250-figd2104-2L-e |    34 |        5 |        2 |         1 |       2 |       16 | <16, 8>        |          1 |     1 |  209 |   564 |      0 |       1131057 |
+----------------------+-------+----------+----------+-----------+---------+----------+----------------+------------+-------+------+-------+--------+---------------+
| xcu250-figd2104-2L-e |    34 |        5 |        2 |         1 |       2 |       16 | <16, 8>        |          1 |     1 |  209 |   564 |      0 |       1131057 |
+---